# Nota del portafolio

Proyecto académico de Santiago Lara. Esta copia conserva el código original y elimina salidas guardadas y conclusiones numéricas no verificadas. No se ha vuelto a ejecutar en la preparación del portafolio porque falta el CSV original. Consultar README para datos, ejecución y limitaciones.

# Proyecto Práctico – Introducción a Ciencia de Datos
## ETL básico y análisis exploratorio para evaluar la posibilidad de crear un modelo predictivo

**Dataset:** Home Loan Approval Dataset (Dream Housing Finance)
**Variable objetivo:** `Loan_Status`
**Pregunta principal:** ¿Con los datos disponibles, es posible crear un modelo predictivo que ayude a estimar si una solicitud de préstamo será aprobada o rechazada?

Este notebook documenta, paso a paso, el flujo completo de trabajo: comprensión del dataset, ETL (Extract-Transform-Load), análisis exploratorio de datos (EDA), visualizaciones, validación de la posibilidad de un modelo predictivo y, de forma opcional, el entrenamiento de un modelo básico de clasificación.

## Etapa 1: Comprensión del dataset

Antes de tocar el código, es necesario entender qué representa la información:

- **Cada fila** del dataset representa **una solicitud de préstamo de vivienda** realizada por un cliente a la empresa Dream Housing Finance.
- **Tipo de problema:** es un problema de **clasificación binaria**, porque la variable que se desea predecir (`Loan_Status`) solo puede tomar dos valores: `Y` (aprobado) o `N` (no aprobado).
- **Variable a predecir:** `Loan_Status`.
- **Variables que podrían ayudar a explicar la aprobación o rechazo:** `Credit_History` (historial crediticio), `ApplicantIncome` e `CoapplicantIncome` (ingresos), `LoanAmount` (monto solicitado), `Education` (nivel educativo), `Married` (estado civil), `Property_Area` (zona de la propiedad), `Self_Employed` (si es independiente), entre otras.
- **Utilidad para una empresa financiera:** un modelo que prediga la probabilidad de aprobación permite a Dream Housing Finance **agilizar el proceso de originación de créditos**, priorizar la revisión manual en los casos dudosos, reducir el riesgo de otorgar préstamos que probablemente no cumplan los criterios internos, y dar una respuesta más rápida y consistente al cliente.

## Etapa 2: ETL básico
### A. Extracción de datos (Extract)

Se carga el archivo CSV descargado de Kaggle utilizando `pandas`, y se realiza una primera inspección: dimensiones, tipos de datos y primeras filas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

# Carga del dataset original (Extract)

df = pd.read_csv("loan_sanction_train.csv")

print(f"El dataset tiene {df.shape[0]} filas y {df.shape[1]} columnas.")

In [ ]:
# Primeras filas del dataset
df.head()

In [ ]:
# Forma del dataset (filas, columnas)
df.shape

In [ ]:
# Información general: tipos de datos y valores no nulos por columna
df.info()

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

### B. Transformación de datos (Transform)

En esta etapa se revisan valores nulos, tipos de datos, variables categóricas y numéricas, columnas irrelevantes y duplicados, y se documenta cada decisión tomada.

In [ ]:
# Revisión de valores nulos por columna
nulos = df.isnull().sum().sort_values(ascending=False)
porcentaje_nulos = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({"valores_nulos": nulos, "porcentaje_%": porcentaje_nulos})
resumen_nulos[resumen_nulos["valores_nulos"] > 0]

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

In [ ]:
# Separación de variables numéricas y categóricas
columnas_numericas = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
columnas_categoricas = df.select_dtypes(include=["object"]).columns.tolist()

print("Variables numéricas:", columnas_numericas)
print("Variables categóricas:", columnas_categoricas)

**Decisión de imputación:**
- Variables **numéricas** (`LoanAmount`, `Loan_Amount_Term`, `Credit_History`): se imputan con la **mediana**, porque es una medida robusta ante valores atípicos (por ejemplo, montos de préstamo muy altos) y no distorsiona la distribución tanto como la media.
- Variables **categóricas** (`Gender`, `Married`, `Dependents`, `Self_Employed`): se imputan con la **moda** (valor más frecuente), ya que es la estrategia estándar para variables cualitativas cuando el porcentaje de nulos es bajo.

In [ ]:
df_limpio = df.copy()

# Imputación de variables numéricas con la mediana
for col in ["LoanAmount", "Loan_Amount_Term", "Credit_History"]:
    mediana = df_limpio[col].median()
    df_limpio[col] = df_limpio[col].fillna(mediana)
    print(f"'{col}': valores nulos rellenados con la mediana = {mediana}")

In [ ]:
# Imputación de variables categóricas con la moda
for col in ["Gender", "Married", "Dependents", "Self_Employed"]:
    moda = df_limpio[col].mode()[0]
    df_limpio[col] = df_limpio[col].fillna(moda)
    print(f"'{col}': valores nulos rellenados con la moda = '{moda}'")

In [ ]:
# Verificación: ya no deben quedar valores nulos
print("Valores nulos restantes por columna:")
print(df_limpio.isnull().sum().sum(), "valores nulos en total")

**Revisión de duplicados**

In [ ]:
duplicados = df_limpio.duplicated().sum()
print(f"Cantidad de filas duplicadas: {duplicados}")

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

**Eliminación de columnas que no aportan al análisis**

`Loan_ID` es un identificador único por solicitud. No tiene relación con la aprobación del préstamo y, si se dejara dentro del modelo, podría generar sobreajuste (el modelo "memorizaría" IDs en vez de aprender patrones reales). Por lo tanto, se elimina como variable predictora, aunque se conserva por separado como referencia.

In [ ]:
loan_ids = df_limpio["Loan_ID"]
df_modelo = df_limpio.drop(columns=["Loan_ID"])
print("Columnas finales disponibles para el análisis y un futuro modelo:")
print(df_modelo.columns.tolist())

**Conversión de variables categóricas**

Para dejar el dataset preparado para un futuro modelo predictivo, se convierten las variables categóricas a formato numérico mediante codificación (label encoding para variables binarias/ordinales simples). Esto no cambia el dataset limpio final entregado (que conserva las categorías originales legibles), sino que se aplica en una copia auxiliar usada más adelante para el modelo opcional.

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_codificado = df_modelo.copy()
codificadores = {}

for col in columnas_categoricas:
    if col == "Loan_ID":
        continue
    le = LabelEncoder()
    df_codificado[col] = le.fit_transform(df_codificado[col].astype(str))
    codificadores[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print("Variables categóricas codificadas:", [c for c in columnas_categoricas if c != "Loan_ID"])
print("\nEjemplo de codificación de Loan_Status:", codificadores["Loan_Status"])

### C. Carga o salida final (Load)

Se guarda el dataset limpio (con las categorías en su forma original y legible, sin `Loan_ID`) como archivo CSV final: `home_loan_clean.csv`.

In [ ]:
# Guardar el dataset limpio en el mismo directorio del notebook

df_final = df_modelo.copy()

df_final.to_csv("home_loan_clean.csv", index=False)

print("Archivo 'home_loan_clean.csv' generado correctamente.")
print(f"Dimensiones finales: {df_final.shape[0]} filas x {df_final.shape[1]} columnas")

df_final.head()

## Etapa 3: Análisis exploratorio de datos (EDA)
### Análisis general

In [ ]:
print(f"Cantidad de registros: {df_modelo.shape[0]}")
print(f"Cantidad de variables: {df_modelo.shape[1]}")
print(f"\nTipos de variables:\n{df_modelo.dtypes}")

In [ ]:
# Estadísticas descriptivas de variables numéricas
df_modelo.describe()

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

### Análisis de la variable objetivo: `Loan_Status`

In [ ]:
conteo_status = df_modelo["Loan_Status"].value_counts()
porcentaje_status = df_modelo["Loan_Status"].value_counts(normalize=True).mul(100).round(1)
print("Conteo de solicitudes por estado:")
print(conteo_status)
print("\nPorcentaje:")
print(porcentaje_status)

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

### Análisis de posibles variables predictoras

In [ ]:
# Relación entre Credit_History y Loan_Status
tabla_credito = pd.crosstab(df_modelo["Credit_History"], df_modelo["Loan_Status"], normalize="index").mul(100).round(1)
print("Porcentaje de aprobación según historial crediticio:")
tabla_credito

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

In [ ]:
# Ingreso promedio del solicitante según el resultado del préstamo
df_modelo.groupby("Loan_Status")["ApplicantIncome"].describe()[["mean", "50%", "std"]]

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

In [ ]:
# Relación entre zona de la propiedad y aprobación
tabla_zona = pd.crosstab(df_modelo["Property_Area"], df_modelo["Loan_Status"], normalize="index").mul(100).round(1)
print("Porcentaje de aprobación según zona de propiedad:")
tabla_zona

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

In [ ]:
# Relación entre educación y aprobación
tabla_educacion = pd.crosstab(df_modelo["Education"], df_modelo["Loan_Status"], normalize="index").mul(100).round(1)
print("Porcentaje de aprobación según nivel educativo:")
tabla_educacion

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

## Etapa 4: Visualizaciones requeridas

A continuación se presentan 6 visualizaciones (más de las 4 mínimas requeridas), cada una con su interpretación.

In [ ]:
# Visualización 1: Distribución de Loan_Status

fig, ax = plt.subplots(figsize=(6, 4))

sns.countplot(
    data=df_modelo,
    x="Loan_Status",
    palette=["#4C72B0", "#DD8452"],
    ax=ax
)

ax.set_title("Distribución de la variable objetivo (Loan_Status)")
ax.set_xlabel("Estado del préstamo")
ax.set_ylabel("Cantidad de solicitudes")

plt.tight_layout()
plt.show()

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

In [ ]:
# Visualización 2: Credit_History vs Loan_Status

fig, ax = plt.subplots(figsize=(6, 4))

sns.countplot(
    data=df_modelo,
    x="Credit_History",
    hue="Loan_Status",
    palette=["#DD8452", "#4C72B0"],
    ax=ax
)

ax.set_title("Historial crediticio vs. estado del préstamo")
ax.set_xlabel("Historial crediticio (0 = negativo, 1 = positivo)")
ax.set_ylabel("Cantidad de solicitudes")

plt.tight_layout()
plt.show()

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

In [ ]:
# Visualización 3: Histograma de ApplicantIncome

fig, ax = plt.subplots(figsize=(6, 4))

sns.histplot(
    df_modelo["ApplicantIncome"],
    bins=40,
    kde=True,
    color="#4C72B0",
    ax=ax
)

ax.set_title("Distribución del ingreso del solicitante")
ax.set_xlabel("Ingreso del solicitante")
ax.set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

In [ ]:
# Visualización 4: Boxplot de LoanAmount

fig, ax = plt.subplots(figsize=(6, 4))

sns.boxplot(
    data=df_modelo,
    y="LoanAmount",
    color="#DD8452",
    ax=ax
)

ax.set_title("Distribución del monto del préstamo solicitado")
ax.set_ylabel("Monto del préstamo (miles)")

plt.tight_layout()
plt.show()

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

In [ ]:
# Visualización 5: Property_Area vs Loan_Status

fig, ax = plt.subplots(figsize=(6, 4))

sns.countplot(
    data=df_modelo,
    x="Property_Area",
    hue="Loan_Status",
    palette=["#DD8452", "#4C72B0"],
    ax=ax
)

ax.set_title("Zona de la propiedad vs. estado del préstamo")
ax.set_xlabel("Zona de la propiedad")
ax.set_ylabel("Cantidad de solicitudes")

plt.tight_layout()
plt.show()

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

In [ ]:
# Visualización 6: Education vs Loan_Status

fig, ax = plt.subplots(figsize=(6, 4))

sns.countplot(
    data=df_modelo,
    x="Education",
    hue="Loan_Status",
    palette=["#DD8452", "#4C72B0"],
    ax=ax
)

ax.set_title("Nivel educativo vs. estado del préstamo")
ax.set_xlabel("Nivel educativo")
ax.set_ylabel("Cantidad de solicitudes")

plt.tight_layout()
plt.show()

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

## Etapa 6 (opcional): Modelo predictivo básico

Se entrena un modelo simple de **regresión logística** para verificar en la práctica si el dataset permite predecir `Loan_Status`. No se busca optimización avanzada, sino comprobar el flujo completo: separar variables, dividir en entrenamiento/prueba, entrenar, evaluar e interpretar.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Variables predictoras (X) y variable objetivo (y)
X = df_codificado.drop(columns=["Loan_Status"])
y = df_codificado["Loan_Status"]

# División en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Registros de entrenamiento: {X_train.shape[0]}")
print(f"Registros de prueba: {X_test.shape[0]}")

In [ ]:
# Entrenamiento del modelo de regresión logística

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

modelo = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000)
)

modelo.fit(X_train, y_train)

# Predicciones sobre el conjunto de prueba
y_pred = modelo.predict(X_test)

In [ ]:
# Evaluación del modelo
exactitud = accuracy_score(y_test, y_pred)
print(f"Exactitud (accuracy) del modelo: {exactitud:.2%}")

matriz = confusion_matrix(y_test, y_pred)
print("\nMatriz de confusión:")
print(matriz)

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=["No aprobado (0)", "Aprobado (1)"]))

In [ ]:
# Visualización de la matriz de confusión

fig, ax = plt.subplots(figsize=(5, 4))

sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicho: No", "Predicho: Sí"],
    yticklabels=["Real: No", "Real: Sí"],
    ax=ax
)

ax.set_title("Matriz de confusión - Regresión Logística")

plt.tight_layout()
plt.show()

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.

## Interpretación y alcance

Revisar las salidas al ejecutar el cuaderno. Este ejercicio es exploratorio; las asociaciones no demuestran causalidad. El modelo estima la aprobación registrada, no el riesgo de impago. La evaluación es preliminar: la imputación y codificación se realizan antes de separar entrenamiento y prueba. Para una evaluación rigurosa deben ajustarse únicamente con entrenamiento.